# ClearSpace — Colab image-generation service (R7)

**Honest status: no real image generation has ever succeeded against
this code.** This notebook is the launcher for Phase 1's
contract-compatible service — everything below has been written and
verified with fakes, never against a real GPU. See
`colab_service/README.md` for the full architecture, the current phase,
and the list of provisional items Phase 2 must still confirm.

## Before you run this

**The ngrok tunnel this notebook opens has no application-level
authentication.** Its only protection is that the URL isn't published —
weak, temporary protection, appropriate only while you are actively,
personally testing. Do not leave this notebook running unattended. See
the "Cleanup" cell at the end.

You will need three secrets set up beforehand — see
`colab_service/README.md`'s "Secrets" section for exactly how, on both
Colab Web and the VS Code Colab extension:

| Secret name | Used for |
|---|---|
| `GITHUB_TOKEN` | Cloning the repository |
| `NGROK_AUTHTOKEN` | Authenticating the ngrok tunnel |
| `NGROK_DOMAIN` | The reserved/static ngrok domain to bind to |

No real secret value is ever written in this notebook's source, printed
by any cell, or stored anywhere other than this session's memory.


## Step 1 — Confirm a GPU is attached

Checked first, before anything else — no point installing multi-GB packages without one.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Colab: Runtime > Change runtime type > "
        "Hardware accelerator > GPU, then Runtime > Restart runtime and "
        "re-run this notebook from the top. (VS Code Colab extension: "
        "make sure you're connected to a GPU-backed runtime.)"
    )

print(f"CUDA GPU available: {torch.cuda.get_device_name(0)}")


## Step 2 — Secrets

Tries Colab Web's Secrets panel first (`google.colab.userdata`). Falls
back to a masked, interactive prompt (`getpass`) if that's unavailable
(the VS Code Colab extension does not always expose it) or a given
secret isn't set there. Typed input is never echoed, logged, or written
to a file — see `_get_secret`'s own docstring below.

In [ ]:
import getpass
import os


def _get_secret(name: str) -> str:
    """Reads a secret by name. Tries Colab Web's Secrets panel first
    (google.colab.userdata) -- the normal path. Falls back to a masked,
    interactive prompt if userdata is unavailable (e.g. the VS Code Colab
    extension, which does not always expose userdata.get()) or the named
    secret isn't set there. The typed value is never echoed, never
    logged, and never written to a file. A blank (empty or
    whitespace-only) value from EITHER path is rejected with a clear
    error rather than silently proceeding -- a blank secret is never a
    valid one."""
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value and value.strip():
            return value
    except Exception:
        pass  # userdata unavailable (e.g. VS Code Colab extension) or secret not set there
    value = getpass.getpass(f"Enter {name} (input hidden, not stored): ")
    if not value or not value.strip():
        raise RuntimeError(f"{name} was blank -- a real, non-blank secret value is required")
    return value


GITHUB_TOKEN = _get_secret("GITHUB_TOKEN")
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
NGROK_DOMAIN = _get_secret("NGROK_DOMAIN")

# Only the two ngrok values are needed as plain environment variables by
# the rest of this notebook / colab_service.config.Settings -- GITHUB_TOKEN
# is used once, transiently, by the clone cell below, and is never stored
# in os.environ at all.
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN

print("Secrets loaded (values not shown).")


## Step 3 — Clone the repository (private-repo safe)

The token is **never embedded in the clone URL** — it is passed as
process-local git configuration (`GIT_CONFIG_COUNT`/`GIT_CONFIG_KEY_0`/
`GIT_CONFIG_VALUE_0`, git's own mechanism for injecting config without a
file), scoped only to this one `subprocess.run(...)` call's own
environment (a filtered copy of `os.environ` that also deliberately
excludes `NGROK_AUTHTOKEN`/`NGROK_DOMAIN`, which have no reason to be
visible to git at all) — never the notebook's global `os.environ`,
never a `.git/config` file, never argv. GitHub HTTPS git uses the
personal access token as the HTTP Basic password
(`Authorization: Basic base64(username:token)`); neither the token nor
the encoded credential is ever printed. Because the token was never in
the URL, the cloned `origin` remote is clean from the very first command
— no rewrite is needed. If `/content/clearspace-fyp` already exists,
this cell refuses to run rather than silently reusing a possibly-stale
checkout or deleting anything automatically — restart the runtime, or
remove the directory yourself first, if you're sure. Either way,
`GITHUB_TOKEN` and the encoded credential are cleared from notebook
memory before the cell exits, on every exit path, including this
refusal. Adds the cloned repository root to `sys.path` so
`colab_service` and its siblings become importable exactly as they are
in this repo.


In [ ]:
import base64
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/yxnmei/clearspace-fyp.git"
REPO_DIR = Path("/content/clearspace-fyp")
GITHUB_USERNAME = "yxnmei"

try:
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} already exists. This notebook never reuses a "
            "possibly-stale checkout and never deletes one automatically -- "
            "either restart the Colab runtime (Runtime > Restart runtime) "
            "for a guaranteed-clean session, or delete that directory "
            "yourself first if you are sure it is safe to discard."
        )

    # GitHub HTTPS git uses the personal access token as the HTTP Basic
    # password. Neither GITHUB_TOKEN nor the encoded credential is ever
    # printed, placed in the clone URL, or passed as a command argument
    # -- both travel only inside this one subprocess's own environment,
    # via GIT_CONFIG_VALUE_0 below.
    credential = base64.b64encode(f"{GITHUB_USERNAME}:{GITHUB_TOKEN}".encode("utf-8")).decode("ascii")

    # Built from a FILTERED copy of os.environ, not the raw dict:
    # NGROK_AUTHTOKEN/NGROK_DOMAIN are already plain env vars by this
    # point (set in the secrets cell above) and have no reason to be
    # visible to the clone subprocess at all.
    clone_env = {k: v for k, v in os.environ.items() if k not in ("NGROK_AUTHTOKEN", "NGROK_DOMAIN")}
    clone_env["GIT_CONFIG_COUNT"] = "1"
    clone_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
    clone_env["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {credential}"

    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True, capture_output=True, text=True, env=clone_env,
        )
    except subprocess.CalledProcessError:
        # Deliberately never prints exc.stderr/exc.stdout, and raises with
        # `from None` -- even though the token is not in argv/URL here,
        # nothing about a git error's own text is trusted not to echo the
        # Authorization header value back in some edge case (e.g. a
        # misconfigured proxy). One fixed, sanitized message only.
        raise RuntimeError(
            "git clone failed -- check repository access and your "
            "GITHUB_TOKEN secret, then try again"
        ) from None
    finally:
        # Discard the temporary token-bearing environment and the
        # encoded credential immediately -- neither is needed again
        # this session.
        del clone_env
        del credential
finally:
    # Cleared on EVERY exit path from this cell, success or failure --
    # including the already-exists check above raising before a clone
    # was ever attempted -- so no failure path leaves the token live in
    # notebook memory.
    GITHUB_TOKEN = None

print(f"Cloned to {REPO_DIR} (origin remote was never given the token -- clean from the start).")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"{REPO_DIR} added to sys.path.")


## Step 4 — Install service dependencies

See `colab_service/requirements.txt` — versions are deliberately unpinned/provisional, see the README.

In [ ]:
get_ipython().system('pip install -q -r {REPO_DIR}/colab_service/requirements.txt')
print("colab_service dependencies installed.")


## Step 5 — Load models

Real, heavy, network-fetching model load — this is the slow step (see
`colab_service/README.md` for rough timing expectations, not yet
measured for real). `/health` cannot report `"ok"` until this cell
finishes successfully — see `colab_service/app.py`'s own docstring.

In [ ]:
import colab_service.app as app_module

print("Loading models -- this can take several minutes on first run (fresh weights download).")
app_module.load_models()
print(
    "Models loaded. base_model:", app_module._state.base_model_id,
    "| controlnet_model:", app_module._state.controlnet_model_id,
)


## Step 6 — Start the server and the ngrok tunnel

Runs uvicorn in a background thread so this cell returns immediately and
does not block the rest of the notebook (or prevent you from running the
cleanup cell later in the same session). Polls the local `/health`
endpoint with **bounded polling** (short, sanitized retries — never a
fixed fire-and-hope wait) until it returns the full compatible `"ok"`
body — a `{"status": "loading"}` response (returned before model
loading finishes) does not count, and polling continues through it.
**Only once that succeeds does this cell open the ngrok tunnel** — the
public URL is never exposed for a service that isn't genuinely ready
yet. Only the public service URL and the (already contract-safe,
non-secret) local health-check body are ever printed — nothing else.


In [ ]:
import threading
import time

import requests
from pyngrok import ngrok

_settings = app_module.config.get_settings()


def _run_server():
    import uvicorn

    uvicorn.run(app_module.app, host=_settings.host, port=_settings.port, log_level="warning")


def _is_compatible_health_body(data: object) -> bool:
    """Mirrors app.models.image_gen_client.check_health()'s exact
    compatibility criteria on the JSON body (status/api_version/
    service_version/capabilities.depth_controlnet) -- see that
    function's own docstring in backend/. The HTTP status-code gate
    (2xx) is checked separately by the caller, _poll_local_health,
    before this is ever called. A {"status": "loading"} body (returned
    by colab_service/app.py's own /health handler before load_models()
    has finished) deliberately does NOT satisfy this, so
    _poll_local_health keeps retrying through it instead of treating a
    not-yet-ready service as ready."""
    if not isinstance(data, dict):
        return False
    if data.get("status") != "ok":
        return False
    if data.get("api_version") != app_module.schemas.API_VERSION:
        return False
    service_version = data.get("service_version")
    if not isinstance(service_version, str) or not service_version.strip():
        return False
    capabilities = data.get("capabilities")
    if not isinstance(capabilities, dict):
        return False
    if capabilities.get("depth_controlnet") is not True:
        return False
    return True


def _poll_local_health(*, attempts: int = 20, delay_s: float = 1.5) -> dict:
    """Bounded polling, never a fixed sleep-and-hope wait -- retries a
    short, fixed number of times with a short delay between attempts,
    tolerating both the brief window before uvicorn has bound its socket
    AND the longer, expected window where /health responds but with
    {"status": "loading"} because load_models() hasn't finished yet.
    Keeps retrying unless ALL of the following hold: the HTTP status is
    2xx, the body parses as JSON, the parsed JSON is an object, and that
    object is genuinely the full compatible "ok" contract shape (see
    _is_compatible_health_body) -- a non-2xx status, invalid JSON, a
    "loading" body, any other incompatible body, or a connection failure
    are all treated as retryable, never printed raw. Raises one clear,
    sanitized error (no raw response body, URL, or exception text) if
    the poll budget is exhausted without ever seeing a compatible
    response."""
    last_state = None
    for _ in range(attempts):
        try:
            response = requests.get(f"http://127.0.0.1:{_settings.port}/health", timeout=5)
        except requests.RequestException as exc:
            last_state = type(exc).__name__
            time.sleep(delay_s)
            continue
        if not (200 <= response.status_code < 300):
            last_state = "non-2xx status"
            time.sleep(delay_s)
            continue
        try:
            data = response.json()
        except ValueError as exc:
            last_state = type(exc).__name__
            time.sleep(delay_s)
            continue
        if _is_compatible_health_body(data):
            return data
        last_state = "incompatible health response"
        time.sleep(delay_s)
    raise RuntimeError(
        f"Local /health did not report a compatible ready body after {attempts} attempts "
        f"({attempts * delay_s:.0f}s) -- last observed state: {last_state}. "
        "The server thread or model loading may have failed; check for errors "
        "printed above this cell."
    )


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Poll the LOCAL health endpoint until the service is genuinely, fully
# ready BEFORE opening the ngrok tunnel -- the public URL must never be
# exposed for a service that isn't ready yet.
local_health = _poll_local_health()

ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
tunnel = ngrok.connect(_settings.port, domain=os.environ["NGROK_DOMAIN"])

print("Service URL:", tunnel.public_url)
print("Local health check:", local_health)


## Cleanup

**Closing this browser tab (or disconnecting the VS Code Colab
extension) is NOT sufficient cleanup on its own** — the Colab runtime,
the uvicorn server, and the ngrok tunnel can all keep running in the
background afterward. Do both of the following explicitly when you are
done testing — do not leave the tunnel running unattended (see the
security note at the top of this notebook):

1. **Stop the tunnel** — uncomment and run the cell below (commented out
   by default so `Runtime > Run all` never stops the service the moment
   it starts).
2. **Terminate the Colab runtime itself**:
   - **Colab Web**: `Runtime > Manage sessions > Terminate`.
   - **VS Code Colab extension**: run the extension's own
     **"Colab: Remove Server"** command from the Command Palette —
     closing the editor tab alone does not do this.

The server thread is a daemon thread and dies once the runtime is
genuinely terminated either way — but until step 2 is done, the runtime
(and the exposed, unauthenticated tunnel) may still be live even with no
tab open.

In [ ]:
# Uncomment and run when you want to stop the tunnel (this alone does
# NOT terminate the Colab runtime -- see step 2 above, which you must
# also do, via the Colab Web or VS Code Colab extension UI):
#
# from pyngrok import ngrok
# ngrok.disconnect(tunnel.public_url)
# ngrok.kill()
